Middleware

- Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

1. Tracking agent behavior with logging, analytics, and debugging.
2. Transforming prompts, tool selection, and output formatting.
3. Adding retries, fallbacks, and early termination logic.
4. Applying rate limits, guardrails, and PII detection.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Summaraisation Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model = "gpt-4.1-mini",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4.1-mini",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]

)
config = {"configurable": {"thread_id":"test-1"}}

In [5]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a6066bec-5bb1-4051-9ab1-27efffc67bf7'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c31e0081d1', 'id': 'chatcmpl-EAHn2PbvXRUwSekWnv6sDy8FZICmu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fdd11-f981-7f21-8bc9-3980bed44653-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read':

In [6]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Q: {q}")
    print(f"A: {response['messages'][-1].content}")
    print(f"Message Count: {len(response['messages'])}")
    print("-" * 50)

Q: What is 2+2?
A: 2 plus 2 equals 4.
Message Count: 8
--------------------------------------------------
Q: What is 10*5?
A: 10 multiplied by 5 equals 50.
Message Count: 10
--------------------------------------------------
Q: What is 100/4?
A: 100 divided by 4 equals 25.
Message Count: 6
--------------------------------------------------
Q: What is 15-7?
A: 15 minus 7 equals 8.
Message Count: 8
--------------------------------------------------
Q: What is 3*3?
A: 3 multiplied by 3 equals 9.
Message Count: 10
--------------------------------------------------
Q: What is 4*4?
A: 4 multiplied by 4 equals 16.
Message Count: 6
--------------------------------------------------


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model = "gpt-4.1-mini",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "gpt-4.1-mini",
            trigger=("tokens",500),
            keep=("tokens",100)
        )
    ]
)
config = {"configurable": {"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return  total_chars // 4 # 4 chars = 1 token


In [ ]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content= f"Find hotels in {city}")]},
        config=config
    )
    
tokens = count_tokens(response["messages"])
print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
print(f"{(response['messages'])}")


Singapore: ~467 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is seeking hotel options in multiple major cities, including Dubai.\n\n## SUMMARY\n\nThe user initially requested hotel listings in Paris, London, Tokyo, and New York. For each city, three consistent hotel options were provided, each with identical names, star ratings, prices, and amenities:\n\n- Grand Hotel: 5 stars, $350/night, amenities include spa, pool, and gym  \n- City Inn: 4 stars, $180/night, with a business center  \n- Budget Stay: 3 stars, $75/night, with free wifi  \n\nThe user then expanded their request to include hotels in Dubai. The same three hotel options with the same attributes were provided for Dubai as well.\n\n## ARTIFACTS\n\n- Search tool calls for hotel listings in Paris, London, Tokyo, New York, and Dubai  \n- Hotel listings for each city featuring identical hotel options with specified star ratings, prices, and amenities\n\

In [9]:
print(f"\n{city}")

for msg in response["messages"]:
        
    if (
        isinstance(msg, HumanMessage)
        and msg.additional_kwargs.get("lc_source") == "summarization"
        ):
        print("📝 SUMMARY")
        print(msg.content)

    else:
        print(type(msg).__name__, msg.content[:60])


Singapore
📝 SUMMARY
Here is a summary of the conversation to date:

## SESSION INTENT

The user is seeking hotel options in multiple major cities, including Dubai.

## SUMMARY

The user initially requested hotel listings in Paris, London, Tokyo, and New York. For each city, three consistent hotel options were provided, each with identical names, star ratings, prices, and amenities:

- Grand Hotel: 5 stars, $350/night, amenities include spa, pool, and gym  
- City Inn: 4 stars, $180/night, with a business center  
- Budget Stay: 3 stars, $75/night, with free wifi  

The user then expanded their request to include hotels in Dubai. The same three hotel options with the same attributes were provided for Dubai as well.

## ARTIFACTS

- Search tool calls for hotel listings in Paris, London, Tokyo, New York, and Dubai  
- Hotel listings for each city featuring identical hotel options with specified star ratings, prices, and amenities

## NEXT STEPS

- Confirm if the user wants additional hot

In [ ]:
#Fractions




# HumanInTheLoopMiddleware
# Check for input and output words /harmful words
# PII Detetction Email ID, Credit Card info, API Key 
# Model Fall Back